# CIL Monocular Depth — Colab Training
**Before running:** Runtime → Change runtime type → A100/G4

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
REPO_URL      = 'git@github.com:xSurus/CIL_Monocular_depth.git'
CONFIG        = 'configs/overgrown_conf_12ep_bs40_data_aug_refiner.yaml'
KAGGLE_TOKEN  = '...'  # paste your token here
SUBMIT        = False
BRANCH        = 'master'
BASE_CHECKPOINT = None  # e.g. '/gdrive/MyDrive/CIL/checkpoints/run_best.pt'; None = use trained best
FORCE_RETRAIN_REFINER  = True  # set True to retrain even if refiner checkpoint exists
FORCE_REBUILD_VEGLIST  = False  # set True to re-run CLIP and rebuild vegetation_list.txt


# ── PATHS — Drive ────────────────────────────────────────────────────────────
DRIVE_BASE        = '/gdrive/MyDrive/CIL'
OVERGROWN_ZIP     = f'{DRIVE_BASE}/data/overgrown_train.zip'
DRIVE_CHECKPOINTS = f'{DRIVE_BASE}/checkpoints'
DRIVE_CLIP        = f'{DRIVE_BASE}/clip_scores'
DRIVE_LOGS        = f'{DRIVE_BASE}/logs'
DRIVE_VIS         = f'{DRIVE_BASE}/visualizations'
DRIVE_SUBMISSIONS = f'{DRIVE_BASE}/submissions'

# ── PATHS — Local (Colab SSD) ────────────────────────────────────────────────
DATA_LOCAL      = '/content/data'
OVERGROWN_LOCAL = f'{DATA_LOCAL}/overgrown_train'
CLIP_LOCAL      = f'{DATA_LOCAL}/clip_scores'
TRAIN_DIR       = f'{DATA_LOCAL}/monodepth_kaggle2026/train'
TEST_DIR        = f'{DATA_LOCAL}/monodepth_kaggle2026/test'
WATER_CSV       = f'{CLIP_LOCAL}/train_water.csv'
CATS_CSV        = f'{CLIP_LOCAL}/test_categories.csv'
VGGT_PRED_DIR   = '/content/test_preds/vggt'
SAM_ZIP       = f'{DRIVE_BASE}/data/test_sam_v2.zip'
TRAIN_SAM_ZIP = f'{DRIVE_BASE}/data/train_sam_v2.zip'
SAM_LOCAL = f'{DATA_LOCAL}/sam'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# 0. Imports
import os, sys, yaml, json

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
# 2. SSH key — first time: generates and saves to Drive; every session: loads from Drive
SSH_KEY_PATH = f'{DRIVE_BASE}/id_ed25519'
if not os.path.exists(SSH_KEY_PATH):
    !mkdir -p /root/.ssh {DRIVE_BASE}
    !ssh-keygen -t ed25519 -C 'colab' -f /root/.ssh/id_ed25519 -N '' -q
    !cp /root/.ssh/id_ed25519 {SSH_KEY_PATH}
    !cp /root/.ssh/id_ed25519.pub {SSH_KEY_PATH}.pub
    print('*** Add this key to github.com/settings/keys, then re-run this cell ***')
    !cat /root/.ssh/id_ed25519.pub
else:
    !mkdir -p /root/.ssh
    !cp {SSH_KEY_PATH} /root/.ssh/id_ed25519
    !cp {SSH_KEY_PATH}.pub /root/.ssh/id_ed25519.pub
    !chmod 600 /root/.ssh/id_ed25519
    !ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
    print('SSH key loaded from Drive.')

In [ ]:
# 3. Clone / pull repo
import importlib
if not os.path.exists('/content/CIL_Monocular_depth'):
    !git clone -b {BRANCH} {REPO_URL} /content/CIL_Monocular_depth
else:
    !git -C /content/CIL_Monocular_depth remote set-url origin {REPO_URL}
    !git -C /content/CIL_Monocular_depth fetch origin
    !git -C /content/CIL_Monocular_depth reset --hard origin/{BRANCH}
%cd /content/CIL_Monocular_depth
sys.path.insert(0, '/content/CIL_Monocular_depth')
importlib.invalidate_caches()
if 'tools.colab_notebook_utils' in sys.modules:
    import tools.colab_notebook_utils as _cnu
    importlib.reload(_cnu)
!pip install -q timm

In [ ]:
# 4. Download dataset from Kaggle + extract overgrown data from Drive → local SSD
from tools.colab_notebook_utils import pull_and_extract_drive_archive

# Dataset via Kaggle API (fast: CDN → local SSD)
if not os.path.exists(f'{DATA_LOCAL}/monodepth_kaggle2026'):
    import os; os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
    !pip install -q kaggle
    !kaggle competitions download -c ethz-cil-monocular-depth-estimation-2026 -p /content/
    !unzip -q /content/ethz-cil-monocular-depth-estimation-2026.zip -d {DATA_LOCAL}
    !rm /content/ethz-cil-monocular-depth-estimation-2026.zip
    print('Dataset ready.')
else:
    print('Dataset already on local SSD.')

pull_and_extract_drive_archive(OVERGROWN_ZIP, OVERGROWN_LOCAL, OVERGROWN_LOCAL, expect_rgb_files=True)
pull_and_extract_drive_archive(SAM_ZIP, SAM_LOCAL, f'{SAM_LOCAL}/test_sam_v2')
pull_and_extract_drive_archive(TRAIN_SAM_ZIP, SAM_LOCAL, f'{SAM_LOCAL}/train_sam_v2')

In [ ]:
# 5. Patch config paths + symlink checkpoints to Drive
import re as _re
def _patch_key(text, key, value):
    return _re.sub(rf'^({key}:\s*).*', rf'\g<1>{value}', text, flags=_re.MULTILINE)

with open(CONFIG) as f:
    _text = f.read()
_cfg = yaml.safe_load(_text)
_text = _patch_key(_text, 'data_dir', f'{DATA_LOCAL}/monodepth_kaggle2026')
if 'filter_aerial_csv' in _cfg:
    _text = _patch_key(_text, 'filter_aerial_csv', f'{CLIP_LOCAL}/train_aerial.csv')
if 'water_csv' in _cfg:
    _text = _patch_key(_text, 'water_csv', WATER_CSV)
if 'extra_train_dir' in _cfg:
    _text = _patch_key(_text, 'extra_train_dir', OVERGROWN_LOCAL)
with open(CONFIG, 'w') as f:
    f.write(_text)
_cfg = yaml.safe_load(_text)
print(f'Config updated: data_dir={_cfg["data_dir"]} batch_size={_cfg["batch_size"]}')

os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
if not os.path.islink('/content/CIL_Monocular_depth/checkpoints'):
    !rm -rf /content/CIL_Monocular_depth/checkpoints
    !ln -s {DRIVE_CHECKPOINTS} /content/CIL_Monocular_depth/checkpoints
print('Checkpoints → Drive')

In [ ]:
# 6. CLIP scene scoring — compute/restore categories via helper function
from tools.clip_scene_scoring import run_clip_scene_scoring

FORCE_RECOMPUTE_VEG = False

run_clip_scene_scoring(
    train_dir=TRAIN_DIR,
    test_dir=TEST_DIR,
    clip_local=CLIP_LOCAL,
    clip_drive=DRIVE_CLIP,
    force_recompute_veg=FORCE_RECOMPUTE_VEG,
)

In [ ]:
# 7. Build vegetation list (for swap logic)
import shutil as _shutil
_veglist_local = '/content/CIL_Monocular_depth/src/vegetation_list.txt'
_veglist_drive = f'{DRIVE_CLIP}/vegetation_list.txt'

if not FORCE_REBUILD_VEGLIST and os.path.exists(_veglist_drive):
    _shutil.copy(_veglist_drive, _veglist_local)
    print(f'Vegetation list restored from Drive ({open(_veglist_local).read().count(chr(10))} images).')
else:
    !PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} tools/build_vegetation_list.py \
        --test_dir {TEST_DIR} --out {_veglist_local}
    _shutil.copy(_veglist_local, _veglist_drive)
    print('Vegetation list saved to Drive.')

In [ ]:
# 8. Train
with open(CONFIG) as f:
    _cfg = yaml.safe_load(f)
_run_name = _cfg.get('name', 'run')

if BASE_CHECKPOINT:
    print(f'BASE_CHECKPOINT set — skipping training, using {BASE_CHECKPOINT}')
else:
    LOG_DIR = DRIVE_LOGS
    os.makedirs(LOG_DIR, exist_ok=True)
    LOG_FILE = f'{LOG_DIR}/{_run_name}.log'
    cmd = f'PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} src/train.py --config {CONFIG}'
    !{cmd} 2>&1 | tee {LOG_FILE}

In [ ]:
# 9. Distribution plots
!PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} tools/plot_distribution.py

In [ ]:
# 10. (Optional) Precompute depth cache
_depth_dir = f'/content/depth_cache/{_run_name}'
_weights   = BASE_CHECKPOINT or f'checkpoints/{_run_name}_best.pt'
from pathlib import Path as _Path
if _Path(_depth_dir).exists() and any(_Path(_depth_dir).glob('*.npy')):
    print('Depth cache exists — skipping.')
else:
    !PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} src/precompute_depth.py --config {CONFIG} --weights {_weights} --out {_depth_dir}

In [ ]:
# 11. (Optional) Train refiner
_cfg = yaml.safe_load(open(CONFIG))
if not _cfg.get('refiner'):
    print('refiner not set — skipping.')
else:
    _refiner_name    = _cfg['refiner']['name']
    _refiner_weights = f'checkpoints/{_refiner_name}_best.pt'
    _depth_dir       = f'/content/depth_cache/{_run_name}'
    from pathlib import Path as _Path
    if _Path(_refiner_weights).exists() and not FORCE_RETRAIN_REFINER:
        print(f'Refiner checkpoint exists — skipping. Set FORCE_RETRAIN_REFINER=True to retrain.')
    else:
        !PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} src/train_refiner.py --config {CONFIG} --depth_dir {_depth_dir} --sam_dir {SAM_LOCAL}/train_sam_v2

In [ ]:
# 12. Base predictions
from tools.colab_notebook_utils import resolve_base_run, ensure_test_predictions

_run_name, _weights = resolve_base_run(CONFIG, base_checkpoint=BASE_CHECKPOINT)
PRED_DIR = ensure_test_predictions(
    config_path=CONFIG,
    weights=str(_weights),
    out_dir=f'/content/test_preds/{_run_name}',
)
print(PRED_DIR)

In [ ]:
# 13. (Optional) VGGT predictions
!pip install -q git+https://github.com/facebookresearch/vggt.git
!PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} tools/predict_vggt.py --data-dir {DATA_LOCAL}/monodepth_kaggle2026 --out {VGGT_PRED_DIR}

In [ ]:
# 14. (Optional) Earlier epoch predictions
from tools.colab_notebook_utils import resolve_swap_epoch, resolve_epoch_run, ensure_test_predictions

_swap_epoch = resolve_swap_epoch(CONFIG)
_run_name, _best_weights, _epoch_weights = resolve_epoch_run(CONFIG)
EPOCH_PRED_DIR = ensure_test_predictions(
    config_path=CONFIG,
    weights=str(_epoch_weights),
    out_dir=f'/content/test_preds/{_run_name}_epoch{_swap_epoch}',
)
print(EPOCH_PRED_DIR)

In [ ]:
# 15. (Optional) Generate refined test predictions
_cfg = yaml.safe_load(open(CONFIG))
if not _cfg.get('refiner'):
    print('refiner not set — skipping.')
else:
    _weights  = BASE_CHECKPOINT or f'checkpoints/{_run_name}_best.pt'
    _refiner  = f"checkpoints/{_cfg['refiner']['name']}_best.pt"
    _cmd = (f'PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} src/predict_refiner.py'
            f' --config {CONFIG}'
            f' --base_weights {_weights} --refiner_weights {_refiner}'
            f' --test_sam_dir {SAM_LOCAL}/test_sam_v2 --out /content/test_preds/{_run_name}_refined')
    !{_cmd}

In [ ]:
# 16. Swap vegetation predictions
from tools.colab_notebook_utils import resolve_swap_dirs, swap_selected_predictions

_base       = f'/content/test_preds/{_run_name}'
_repl, _out = resolve_swap_dirs(CONFIG, _base, VGGT_PRED_DIR)

if _repl is None:
    print('No swap configured — skipping.')
elif not os.path.exists(_repl):
    print(f'Replacement not found: {_repl}')
else:
    swap_selected_predictions(base_dir=_base, replacement_dir=_repl,
                              filelist='src/vegetation_list.txt', out_dir=_out)
    print(f'Swap done → {_out}')

In [ ]:
# 17. Build submission CSV
from tools.colab_notebook_utils import build_submission_csv

_submit_variant = yaml.safe_load(open(CONFIG)).get('swap') or 'auto'
SUB_CSV = build_submission_csv(
    config_path=CONFIG,
    drive_submissions=DRIVE_SUBMISSIONS,
    submit_variant=_submit_variant,
    base_checkpoint=BASE_CHECKPOINT,
)

In [ ]:
# 18. Visualize final predictions
from IPython.display import Image, display
from pathlib import Path

_out = f'{DRIVE_VIS}/{Path(SUB_CSV).stem}_preview.png'
!PYTHONPATH=/content/CIL_Monocular_depth {sys.executable} tools/visualize_predictions.py \
    --pred-dir /content/test_preds/{Path(SUB_CSV).stem} \
    --test-dir {TEST_DIR} \
    --out {_out}
display(Image(_out))

In [ ]:
# 19. Submit to Kaggle
from tools.colab_notebook_utils import kaggle_submit

if SUBMIT:
    kaggle_submit(sub_csv=SUB_CSV, kaggle_token=KAGGLE_TOKEN)
else:
    print('SUBMIT=False — skipping. Set SUBMIT=True in cell 1 to run.')